In [ ]:
# scripts/02_embedder.ipynb

import sys
import time
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

# 1. Setup Environment
# Check for GPU (CUDA) or MPS (Mac M1/M2) or CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Running benchmark on: {device.upper()}")

# 2. Define Candidates
# We test 3 distinct classes of models
models_to_test = {
    "MiniLM (Speed)": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-Small (Balanced)": "BAAI/bge-small-en-v1.5",
    "MPNet (Quality)": "sentence-transformers/all-mpnet-base-v2"
}

# 3. Create Benchmark Data
# 1000 sentences to stress-test the batching
base_sentences = [
    "The U.S. economy grew by 2.5% last quarter.",
    "Inflation is currently sitting at 3.2%.",
    "The tech sector saw a massive sell-off today.",
    "Python is a great language for data science.",
    "Apples are generally red or green."
] * 200 

print(f"Test Dataset: {len(base_sentences)} sentences.")

# 4. Semantic Sanity Check Data
# We want a model that sees "bank" (money) and "bank" (river) as DIFFERENT.
s1 = "The bank is closed on Sundays."
s2 = "He sat on the bank of the river." # Homonym (should be distinct from s1)
s3 = "Financial institutions are not open today." # Synonym (should be close to s1)

results = []

# 5. Execution Loop
for friendly_name, model_name in models_to_test.items():
    print(f"\n--- Testing {friendly_name} ---")
    
    # A. Load Time
    start_load = time.time()
    try:
        model = SentenceTransformer(model_name, device=device)
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")
        continue
    load_time = time.time() - start_load
    
    # B. Encoding Speed (Batch Size 32)
    start_encode = time.time()
    # normalize_embeddings=True is CRITICAL for dot-product to equal cosine similarity
    embeddings = model.encode(base_sentences, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    total_encode_time = time.time() - start_encode
    
    sentences_per_sec = len(base_sentences) / total_encode_time
    dim = embeddings.shape[1]
    
    # C. Semantic Quality Test
    e1 = model.encode(s1, convert_to_tensor=True)
    e2 = model.encode(s2, convert_to_tensor=True)
    e3 = model.encode(s3, convert_to_tensor=True)
    
    # Sim(Bank_Money, Bank_River) -> Should be LOW
    sim_homonym = util.cos_sim(e1, e2).item()
    # Sim(Bank_Money, Financial_Inst) -> Should be HIGH
    sim_synonym = util.cos_sim(e1, e3).item()
    # Separation Score (Higher is better)
    separation = sim_synonym - sim_homonym

    print(f"  > Dimensions: {dim}")
    print(f"  > Speed: {sentences_per_sec:.2f} sent/sec")
    print(f"  > Separation Score: {separation:.4f} (Synonym {sim_synonym:.2f} - Homonym {sim_homonym:.2f})")
    
    results.append({
        "Model": friendly_name,
        "Dimensions": dim,
        "Speed (sent/sec)": round(sentences_per_sec, 2),
        "Load Time (s)": round(load_time, 2),
        "Semantic Sep.": round(separation, 3),
        "Synonym Sim": round(sim_synonym, 3),
        "Homonym Sim": round(sim_homonym, 3)
    })

# 6. Final Report
print("\n" + "="*60)
df = pd.DataFrame(results)
# Sort by 'Semantic Sep.' to see who understands language best
display(df.sort_values(by="Semantic Sep.", ascending=False))
print("Recommendation: Pick the fastest model that has acceptable 'Semantic Sep.' (> 0.4)")